In [1]:
import pandas as pd
import numpy as np

In [6]:
df=pd.read_csv("DateFruit_Dataset.csv")
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [8]:
x=df.drop(columns=["Class"])
y=df["Class"]

In [25]:
from sklearn.preprocessing import StandardScaler,LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)

In [26]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [27]:

scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [35]:
x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)

In [36]:
train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)

In [37]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [48]:
class ANN_classifier(nn.Module):
  def __init__(self):
    super(ANN_classifier,self).__init__()
    self.model=nn.Sequential(
        nn.Linear(x_train.shape[1],64),
        nn.ReLU(),

        nn.Linear(64,64),
        nn.ReLU(),

        nn.Linear(64,7),

    )

  def forward(self,x):
    return self.model(x)


In [51]:
model=ANN_classifier()
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [53]:
epochs=100

for epoch in range(epochs):
  model.train()
  running_loss=0.0

  #forward propogation

  for xb,yb in train_loader:
    optimizer.zero_grad()
    output=model(xb)
    loss=criteria(output,yb)

  # backward_propogation
    loss.backward()

    optimizer.step() #params update

    running_loss+=loss
  train_loss=running_loss/len(train_loader)
  print(f"Epoch {epoch+1}/{epochs},Train Loss:{train_loss:.4f}")



Epoch 1/100,Train Loss:1.6549
Epoch 2/100,Train Loss:1.0405
Epoch 3/100,Train Loss:0.6934
Epoch 4/100,Train Loss:0.5230
Epoch 5/100,Train Loss:0.4461
Epoch 6/100,Train Loss:0.3818
Epoch 7/100,Train Loss:0.3475
Epoch 8/100,Train Loss:0.3140
Epoch 9/100,Train Loss:0.2881
Epoch 10/100,Train Loss:0.2657
Epoch 11/100,Train Loss:0.2502
Epoch 12/100,Train Loss:0.2362
Epoch 13/100,Train Loss:0.2285
Epoch 14/100,Train Loss:0.2100
Epoch 15/100,Train Loss:0.2011
Epoch 16/100,Train Loss:0.1880
Epoch 17/100,Train Loss:0.1898
Epoch 18/100,Train Loss:0.1769
Epoch 19/100,Train Loss:0.1737
Epoch 20/100,Train Loss:0.1655
Epoch 21/100,Train Loss:0.1587
Epoch 22/100,Train Loss:0.1580
Epoch 23/100,Train Loss:0.1499
Epoch 24/100,Train Loss:0.1423
Epoch 25/100,Train Loss:0.1415
Epoch 26/100,Train Loss:0.1420
Epoch 27/100,Train Loss:0.1305
Epoch 28/100,Train Loss:0.1293
Epoch 29/100,Train Loss:0.1273
Epoch 30/100,Train Loss:0.1229
Epoch 31/100,Train Loss:0.1192
Epoch 32/100,Train Loss:0.1153
Epoch 33/100,Trai

In [59]:
# evaluation

model.eval()

total=0
correct=0

with torch.no_grad():
  for xb,yb in test_loader:
    output=model(xb)
    _,predicted_val=torch.max(output,1)#give max_val and index

    correct+=(predicted_val==yb).sum().item()
    total+=yb.size(0)

print("total_val",total)
print("correct_val",correct)
print("accuracy",correct/total*100)







total_val 180
correct_val 172
accuracy 95.55555555555556
